# 01. Controlled Data Audit: Raw Datasets
**Project:** SIH - Carbon Market Intelligence & Prediction System  
**Objective:** Controlled audit and inventory of all raw datasets under `data/raw/` without modifying any raw files, moving files, or loading unnecessary dependency folders.  

### Audit Scope & Requirements
For every discovered CSV and XLSX dataset, this audit captures:
1. Relative file path
2. File type
3. Number of rows and columns
4. Column names
5. Data types
6. Missing-value count per column
7. Duplicate-row count
8. Date/year columns and their ranges
9. Approximate unique-value counts for important categorical columns
10. Units/source information
11. Potential role in project (Global market forecasting, Country/regional intelligence, Carbon pricing/policy, Renewable-energy features, Economic features, Company-level carbon trading, Supporting/reference data, Unknown)
12. Data-quality concerns
13. Recommendation (KEEP, KEEP AS SUPPORTING DATA, DO NOT USE FOR ML, INVESTIGATE)

Additionally, datasets are classified by **temporal & spatial granularity** to prevent blind merging.

In [1]:
import os
import sys
import json
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import openpyxl
from openpyxl.styles.fills import PatternFill

# Compatibility patch: Handle openpyxl issue with empty <fill/> XML elements
# (Present in some OECD / official spreadsheet exports)
orig_from_tree = openpyxl.styles.fills.Fill.from_tree

@classmethod
def patched_from_tree(cls, el):
    res = orig_from_tree(el)
    return PatternFill() if res is None else res

openpyxl.styles.fills.Fill.from_tree = patched_from_tree

# Resolve Project Root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
SUPPORTED_EXTENSIONS = {".csv", ".xlsx"}

# Excluded dependency directories
EXCLUDED_DIRS = {".venv", "venv", "node_modules", "site-packages", ".git", ".ipynb_checkpoints"}

print(f"Project Root: {PROJECT_ROOT}")
print(f"Raw Data Dir: {RAW_DATA_DIR}")
print(f"Python: {sys.version.split()[0]} | Pandas: {pd.__version__} | NumPy: {np.__version__} | Openpyxl: {openpyxl.__version__}")

Project Root: C:\Users\ADI\Downloads\carbon-market-intelligence\carbon-market-intelligence
Raw Data Dir: C:\Users\ADI\Downloads\carbon-market-intelligence\carbon-market-intelligence\data\raw
Python: 3.12.1 | Pandas: 3.0.5 | NumPy: 2.5.2 | Openpyxl: 3.1.5


## 1. Controlled Dataset Discovery
Recursively search only within `data/raw/` for `.csv` and `.xlsx` files, ignoring all dependency and cache directories.

In [2]:
def discover_raw_files(raw_dir: Path) -> list[Path]:
    """Discover supported files strictly within raw data dir, skipping excluded paths."""
    if not raw_dir.exists():
        raise FileNotFoundError(f"Raw data directory not found: {raw_dir}")
    discovered = []
    for path in sorted(raw_dir.rglob("*")):
        if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS:
            # Safety check against dependency paths
            if not any(part in EXCLUDED_DIRS for part in path.parts):
                discovered.append(path)
    return discovered

RAW_FILES = discover_raw_files(RAW_DATA_DIR)
print(f"Total supported files discovered: {len(RAW_FILES)}")
for idx, p in enumerate(RAW_FILES, 1):
    print(f"{idx:02d}. [{p.suffix.lower()}] {p.relative_to(PROJECT_ROOT).as_posix()}")

Total supported files discovered: 24
01. [.csv] data/raw/Carbon Trading Transactions Dataset/carbon_trading_dataset.csv
02. [.csv] data/raw/Emissions by Country/GCB2022v27_MtCO2_flat.csv
03. [.csv] data/raw/Emissions by Country/GCB2022v27_percapita_flat.csv
04. [.csv] data/raw/Emissions by Country/GCB2022v27_sources_flat.csv
05. [.xlsx] data/raw/OECD.CTP.TPS,DSD_NECR@DF_NECRS,,filtered,2026-09-02 02-20-27.xlsx
06. [.csv] data/raw/Renewable Energy World Wide  1965~2022/01 renewable-share-energy.csv
07. [.csv] data/raw/Renewable Energy World Wide  1965~2022/02 modern-renewable-energy-consumption.csv
08. [.csv] data/raw/Renewable Energy World Wide  1965~2022/03 modern-renewable-prod.csv
09. [.csv] data/raw/Renewable Energy World Wide  1965~2022/04 share-electricity-renewables.csv
10. [.csv] data/raw/Renewable Energy World Wide  1965~2022/05 hydropower-consumption.csv
11. [.csv] data/raw/Renewable Energy World Wide  1965~2022/06 hydro-share-energy.csv
12. [.csv] data/raw/Renewable Energy W

## 2. Audit Functions & Heuristics
Define metadata mapping, domain role classification, granularity identification, and robust file readers.

In [3]:
# Pre-classified project domain heuristics based on verified file contents
DATASET_METADATA_REGISTRY = {
    "carbon_trading_dataset.csv": {
        "role": "Company-level carbon trading",
        "granularity": "Company-transaction (daily 2024-2025)",
        "units": "Energy: MWh; Emissions/Allowances/Credits: tCO2; Carbon Price: USD/t; Costs: USD",
        "source": "Simulated company trading & compliance transaction dataset",
        "quality_concerns": "Synthetic dataset; perfectly complete (0 nulls); future simulated dates (2024-2025); company-level granularity must not be merged directly with macroeconomic/country series.",
        "recommendation": "KEEP"
    },
    "GCB2022v27_MtCO2_flat.csv": {
        "role": "Country/regional intelligence",
        "granularity": "Country-year panel (1750-2021)",
        "units": "MtCO2 (million metric tonnes CO2); Per Capita in tCO2/person",
        "source": "Global Carbon Project (GCB 2022 v27) / Friedlingstein et al.",
        "quality_concerns": "Extensive missing values prior to 1950; contains regional entities (e.g. 'World', 'Africa') mixed with sovereign nations; 'Other' column has >97% nulls.",
        "recommendation": "KEEP"
    },
    "GCB2022v27_percapita_flat.csv": {
        "role": "Country/regional intelligence",
        "granularity": "Country-year panel (1750-2021)",
        "units": "Tonnes CO2 per person",
        "source": "Global Carbon Project (GCB 2022 v27)",
        "quality_concerns": "High historical missingness (>70% nulls); redundant with 'Per Capita' column already in GCB2022v27_MtCO2_flat.csv.",
        "recommendation": "KEEP AS SUPPORTING DATA"
    },
    "GCB2022v27_sources_flat.csv": {
        "role": "Supporting/reference data",
        "granularity": "Metadata / Provenance table (1750-2021)",
        "units": "Citation strings (CDIAC, BP, UNFCCC, etc.)",
        "source": "Global Carbon Project (GCB 2022 v27)",
        "quality_concerns": "Non-numerical metadata; strictly provenance citations.",
        "recommendation": "DO NOT USE FOR ML"
    },
    "OECD.CTP.TPS,DSD_NECR@DF_NECRS,,filtered,2026-09-02 02-20-27.xlsx": {
        "role": "Carbon pricing/policy",
        "granularity": "Country cross-sectional snapshot (Year 2023)",
        "units": "EUR per tCO2e and National Currency per tCO2e (constant 2023 prices)",
        "source": "OECD Carbon Pricing and Energy Taxation (CPET) Database",
        "quality_concerns": "Openpyxl empty-fill styling quirk; hierarchical header requiring custom skip-rows; single snapshot year (2023) means it cannot serve as longitudinal time-series without static feature mapping.",
        "recommendation": "KEEP AS SUPPORTING DATA"
    },
    "world_gdp_data.csv": {
        "role": "Economic features",
        "granularity": "Country-year wide format (1980-2024)",
        "units": "Annual GDP growth (percent change)",
        "source": "International Monetary Fund (IMF) WEO / World Bank",
        "quality_concerns": "Requires Latin1/cp1252 encoding (fails under standard UTF-8); wide format requires unpivoting (melting) to long format; country names need harmonization with ISO alpha-3.",
        "recommendation": "KEEP"
    },
    "voluntary-carbon-market-size-by-value-and-volume-of-traded-carbon-credits.xlsx": {
        "role": "Global market forecasting",
        "granularity": "Global annual aggregate (pre-2005 to 2024)",
        "units": "Annual/Cumulative value in $M (Million USD); Annual/Cumulative volume in MtCO2e",
        "source": "Ecosystem Marketplace / State of the Voluntary Carbon Markets",
        "quality_concerns": "Small sample size (21 rows); contains 'pre-2005' non-numeric year aggregate; global total only (no country or sectoral breakdown); rows not sorted chronologically in raw file.",
        "recommendation": "KEEP"
    }
}

# Renewable energy dataset metadata generator
def get_renewable_metadata(filename: str) -> dict:
    primary_aggregates = {"03 modern-renewable-prod.csv", "04 share-electricity-renewables.csv"}
    is_primary = filename in primary_aggregates
    return {
        "role": "Renewable-energy features",
        "granularity": "Country-year panel (1965-2022)",
        "units": "TWh (generation/consumption), GW (installed capacity), or % (energy/electricity share)",
        "source": "Our World in Data (OWID) / Energy Institute Statistical Review of World Energy / Ember",
        "quality_concerns": "Redundant metrics across the 17 individual files; capacity metrics cover fewer countries (~33-64) than production metrics (~250); regional entity aggregates mixed with country codes.",
        "recommendation": "KEEP" if is_primary else "KEEP AS SUPPORTING DATA"
    }

def resolve_metadata(filename: str) -> dict:
    if filename in DATASET_METADATA_REGISTRY:
        return DATASET_METADATA_REGISTRY[filename]
    if "renewable" in filename.lower() or any(term in filename.lower() for term in ["solar", "wind", "hydro", "biofuel", "geothermal"]):
        return get_renewable_metadata(filename)
    return {
        "role": "Unknown",
        "granularity": "Unknown",
        "units": "Not specified",
        "source": "Unknown",
        "quality_concerns": "Unclassified file",
        "recommendation": "INVESTIGATE"
    }

def safe_read_data(path: Path) -> tuple[dict[str, pd.DataFrame], str | None]:
    """Read CSV or XLSX safely, returning a dict of {sheet_or_name: DataFrame} and encoding info."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        try:
            df = pd.read_csv(path)
            return {"data": df}, "utf-8"
        except UnicodeDecodeError:
            df = pd.read_csv(path, encoding="latin1")
            return {"data": df}, "latin1"
    elif suffix == ".xlsx":
        xl = pd.ExcelFile(path)
        sheets = {}
        for sheet_name in xl.sheet_names:
            sheets[sheet_name] = pd.read_excel(path, sheet_name=sheet_name)
        return sheets, "xlsx-openpyxl"
    else:
        raise ValueError(f"Unsupported format: {suffix}")

def detect_temporal_range(df: pd.DataFrame) -> dict:
    """Identify date or year columns and compute their range."""
    res = {}
    date_keywords = ("year", "date", "time", "period")
    time_cols = [c for c in df.columns if any(k in str(c).lower() for k in date_keywords)]
    
    # Also check if wide-format years exist as columns (e.g. 1980..2024)
    numeric_cols = [c for c in df.columns if str(c).isdigit() and len(str(c)) == 4 and 1900 <= int(str(c)) <= 2100]
    if len(numeric_cols) > 5:
        res["wide_year_range"] = f"{min(numeric_cols)} - {max(numeric_cols)} (wide columns: {len(numeric_cols)} years)"
    
    for col in time_cols:
        s = df[col].dropna()
        if s.empty:
            continue
        if "year" in str(col).lower() or "date" in str(col).lower() or "time" in str(col).lower():
            res[str(col)] = f"{s.min()} to {s.max()}"
    return res

def get_important_categorical_uniques(df: pd.DataFrame) -> dict:
    """Extract unique counts for key categorical fields (Country, Entity, Fuel, Industry, etc.)."""
    uniques = {}
    for col in df.columns:
        if df[col].dtype == "object" or str(df[col].dtype) == "string":
            n_uniq = df[col].nunique(dropna=True)
            if n_uniq <= 300:
                uniques[str(col)] = n_uniq
    return uniques

## 3. Execute Controlled Audit Across All Discovered Datasets
Perform in-depth analysis on every dataset, strictly isolating errors if any occur.

In [4]:
AUDIT_SUMMARIES = []
COLUMN_AUDIT_RECORDS = []
AUDIT_ERRORS = []

for file_path in RAW_FILES:
    rel_path = file_path.relative_to(PROJECT_ROOT).as_posix()
    fname = file_path.name
    meta = resolve_metadata(fname)
    
    try:
        tables_dict, enc = safe_read_data(file_path)
        
        for table_key, df in tables_dict.items():
            table_label = rel_path if len(tables_dict) == 1 else f"{rel_path} [{table_key}]"
            time_ranges = detect_temporal_range(df)
            cat_uniques = get_important_categorical_uniques(df)
            
            time_range_str = "; ".join(f"{k}: {v}" for k, v in time_ranges.items()) if time_ranges else "None detected"
            
            summary_item = {
                "file_path": table_label,
                "file_type": file_path.suffix.lower().lstrip("."),
                "sheet": table_key if len(tables_dict) > 1 else "N/A",
                "encoding": enc,
                "rows": len(df),
                "cols": len(df.columns),
                "duplicates": int(df.duplicated().sum()),
                "total_missing_cells": int(df.isna().sum().sum()),
                "temporal_coverage": time_range_str,
                "role": meta["role"],
                "granularity": meta["granularity"],
                "recommendation": meta["recommendation"],
                "units": meta["units"],
                "source": meta["source"],
                "quality_concerns": meta["quality_concerns"],
                "column_names": list(map(str, df.columns)),
                "categorical_uniques": cat_uniques
            }
            AUDIT_SUMMARIES.append(summary_item)
            
            # Column-level breakdown
            for col in df.columns:
                col_str = str(col)
                n_miss = int(df[col].isna().sum())
                pct_miss = round((n_miss / len(df) * 100), 2) if len(df) > 0 else 0.0
                COLUMN_AUDIT_RECORDS.append({
                    "file_path": table_label,
                    "column": col_str,
                    "dtype": str(df[col].dtype),
                    "missing_count": n_miss,
                    "missing_pct": pct_miss,
                    "unique_values": int(df[col].nunique(dropna=True))
                })
    except Exception as err:
        AUDIT_ERRORS.append({
            "file_path": rel_path,
            "error": str(err),
            "error_type": type(err).__name__
        })

summary_df = pd.DataFrame(AUDIT_SUMMARIES)
column_audit_df = pd.DataFrame(COLUMN_AUDIT_RECORDS)
errors_df = pd.DataFrame(AUDIT_ERRORS)

print(f"Audited {len(RAW_FILES)} file(s). Produced {len(summary_df)} table records.")
print(f"Total column records tracked: {len(column_audit_df)}")
print(f"Audit execution failures: {len(errors_df)}")

Audited 24 file(s). Produced 25 table records.
Total column records tracked: 189
Audit execution failures: 0


## 4. Master Dataset Audit Inventory
Overview table summarizing every raw dataset, its dimensions, temporal coverage, project role, data-quality concerns, and recommendation.

In [5]:
display_cols = [
    "file_path", "file_type", "rows", "cols", "duplicates",
    "temporal_coverage", "role", "granularity", "recommendation"
]
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 1000)

summary_df[display_cols]

,file_path,file_type,rows,cols,duplicates,temporal_coverage,role,granularity,recommendation
0,data/raw/Carbon Trading Transactions Dataset/carbon_trading_dataset.csv,csv,5000,15,0,Date: 2024-01-01 to 2025-12-31,Company-level carbon trading,Company-transaction (daily 2024-2025),KEEP
1,data/raw/Emissions by Country/GCB2022v27_MtCO2_flat.csv,csv,63104,11,0,Year: 1750 to 2021,Country/regional intelligence,Country-year panel (1750-2021),KEEP
2,data/raw/Emissions by Country/GCB2022v27_percapita_flat.csv,csv,63104,10,0,Year: 1750 to 2021,Country/regional intelligence,Country-year panel (1750-2021),KEEP AS SUPPORTING DATA
3,data/raw/Emissions by Country/GCB2022v27_sources_flat.csv,csv,63104,11,0,Year: 1750 to 2021,Supporting/reference data,Metadata / Provenance table (1750-2021),DO NOT USE FOR ML
4,"data/raw/OECD.CTP.TPS,DSD_NECR@DF_NECRS,,filtered,2026-09-02 02-20-27.xlsx [Table]",xlsx,173,14,1,None detected,Carbon pricing/policy,Country cross-sectional snapshot (Year 2023),KEEP AS SUPPORTING DATA
5,"data/raw/OECD.CTP.TPS,DSD_NECR@DF_NECRS,,filtered,2026-09-02 02-20-27.xlsx [Overview]",xlsx,12,2,4,None detected,Carbon pricing/policy,Country cross-sectional snapshot (Year 2023),KEEP AS SUPPORTING DATA
6,data/raw/Renewable Energy World Wide 1965~2022/01 renewable-share-energy.csv,csv,5603,4,0,Year: 1965 to 2021,Renewable-energy features,Country-year panel (1965-2022),KEEP AS SUPPORTING DATA
7,data/raw/Renewable Energy World Wide 1965~2022/02 modern-renewable-energy-consumption.csv,csv,5610,7,0,Year: 1965 to 2021,Renewable-energy features,Country-year panel (1965-2022),KEEP AS SUPPORTING DATA
8,data/raw/Renewable Energy World Wide 1965~2022/03 modern-renewable-prod.csv,csv,8851,7,0,Year: 1965 to 2022,Renewable-energy features,Country-year panel (1965-2022),KEEP
9,data/raw/Renewable Energy World Wide 1965~2022/04 share-electricity-renewables.csv,csv,6871,4,0,Year: 1985 to 2022,Renewable-energy features,Country-year panel (1965-2022),KEEP


## 5. Granularity & Compatibility Analysis
### ⚠️ WARNING: DO NOT Blindly Merge Incompatible Granularities!
The repository contains datasets spanning vastly different spatial and temporal levels of aggregation.
Merging these naively will create data leakage, false correlations, or massive Cartesian products.

| Granularity Tier | Characteristic | Example Datasets | Proper Handling Strategy |
|---|---|---|---|
| **Tier 1: Global Annual Time-Series** | Single global aggregate row per calendar year | `voluntary-carbon-market-size-by-value-and-volume-of-traded-carbon-credits.xlsx` | Use for macro market sizing & global carbon credit price trend forecasting. Cannot be merged 1:1 with country panel without broadcasting global state. |
| **Tier 2: Country-Year Panel** | Longitudinal panel (Country $\times$ Year) | `GCB2022v27_MtCO2_flat.csv`, `01-17 Renewable Energy CSVs`, `world_gdp_data.csv` (after melt) | Canonical join key: `(ISO_Country_Code, Year)`. Harmonize entity names to ISO 3166-1 alpha-3 and filter to common years (e.g. 1990–2021). |
| **Tier 3: Country Cross-Sectional Snapshot** | Single snapshot year (2023) across countries | `OECD.CTP.TPS...xlsx` | Static country-level policy features (carbon tax rates, ETS coverage). Do not join as a dynamic time-series. |
| **Tier 4: Company-Transaction Sub-Annual** | Daily transaction logs per enterprise (2024–2025) | `carbon_trading_dataset.csv` | Train company-level trading recommendation, allowance compliance, and cost optimization models. Keep distinct from country macro-forecasting. |
| **Tier 5: Provenance & Reference Metadata** | Citation strings & documentation | `GCB2022v27_sources_flat.csv`, `OECD Overview` | Do not use for ML training. Keep strictly for auditability and citations. |

In [6]:
# Granularity grouping summary
granularity_summary = summary_df.groupby(["granularity", "role", "recommendation"]).agg(
    dataset_count=("file_path", "count"),
    total_rows=("rows", "sum"),
    sample_datasets=("file_path", lambda s: list(s)[:2])
).reset_index()

granularity_summary

,granularity,role,recommendation,dataset_count,total_rows,sample_datasets
0,Company-transaction (daily 2024-2025),Company-level carbon trading,KEEP,1,5000,[data/raw/Carbon Trading Transactions Dataset/carbon_trading_dataset.csv]
1,Country cross-sectional snapshot (Year 2023),Carbon pricing/policy,KEEP AS SUPPORTING DATA,2,185,"[data/raw/OECD.CTP.TPS,DSD_NECR@DF_NECRS,,filtered,2026-09-02 02-20-27.xlsx [Table], data/raw/OECD.CTP.TPS,DSD_NECR@DF_NECRS,,filtered,2026-09-02 02-20-27.xlsx [Overview]]"
2,Country-year panel (1750-2021),Country/regional intelligence,KEEP,1,63104,[data/raw/Emissions by Country/GCB2022v27_MtCO2_flat.csv]
3,Country-year panel (1750-2021),Country/regional intelligence,KEEP AS SUPPORTING DATA,1,63104,[data/raw/Emissions by Country/GCB2022v27_percapita_flat.csv]
4,Country-year panel (1965-2022),Renewable-energy features,KEEP,2,15722,"[data/raw/Renewable Energy World Wide 1965~2022/03 modern-renewable-prod.csv, data/raw/Renewable Energy World Wide 1965~2022/04 share-electricity-renewables.csv]"
5,Country-year panel (1965-2022),Renewable-energy features,KEEP AS SUPPORTING DATA,15,79607,"[data/raw/Renewable Energy World Wide 1965~2022/01 renewable-share-energy.csv, data/raw/Renewable Energy World Wide 1965~2022/02 modern-renewable-energy-consumption.csv]"
6,Country-year wide format (1980-2024),Economic features,KEEP,1,196,[data/raw/World GDP Growth/world_gdp_data.csv]
7,Global annual aggregate (pre-2005 to 2024),Global market forecasting,KEEP,1,21,[data/raw/voluntary-carbon-market-size-by-value-and-volume-of-traded-carbon-credits.xlsx]
8,Metadata / Provenance table (1750-2021),Supporting/reference data,DO NOT USE FOR ML,1,63104,[data/raw/Emissions by Country/GCB2022v27_sources_flat.csv]


## 6. Detailed Dataset-by-Dataset Audit Report
Complete inspection report for each of the 24 discovered files detailing all 13 required audit dimensions.

In [7]:
for idx, row in summary_df.iterrows():
    print("=" * 95)
    print(f"DATASET #{idx + 1:02d}: {row['file_path']}")
    print("=" * 95)
    print(f"1. File Type:              {row['file_type'].upper()} (Encoding: {row['encoding']})")
    print(f"2. Dimensions:             {row['rows']:,} rows x {row['cols']} columns")
    print(f"3. Duplicate Rows:         {row['duplicates']:,}")
    print(f"4. Total Missing Cells:    {row['total_missing_cells']:,}")
    print(f"5. Temporal Coverage:      {row['temporal_coverage']}")
    print(f"6. Categorical Uniques:    {row['categorical_uniques']}")
    print(f"7. Units Information:      {row['units']}")
    print(f"8. Source / Provenance:    {row['source']}")
    print(f"9. Project Role:           {row['role']}")
    print(f"10. Granularity:           {row['granularity']}")
    print(f"11. Data Quality Concerns: {row['quality_concerns']}")
    print(f"12. Recommendation:       {row['recommendation']}")
    print(f"13. Column Names ({len(row['column_names'])}): {row['column_names'][:8]}{' ...' if len(row['column_names']) > 8 else ''}")
    print()
    
    # Show column-level slice
    cols_subset = column_audit_df[column_audit_df["file_path"] == row["file_path"]]
    if not cols_subset.empty:
        print("    [Column Details]")
        for _, c_row in cols_subset.head(6).iterrows():
            print(f"      - {c_row['column']} ({c_row['dtype']}): {c_row['missing_count']} nulls ({c_row['missing_pct']}%), {c_row['unique_values']} unique values")
        if len(cols_subset) > 6:
            print(f"      ... and {len(cols_subset) - 6} more columns.")
    print()

DATASET #01: data/raw/Carbon Trading Transactions Dataset/carbon_trading_dataset.csv
1. File Type:              CSV (Encoding: utf-8)
2. Dimensions:             5,000 rows x 15 columns
3. Duplicate Rows:         0
4. Total Missing Cells:    0
5. Temporal Coverage:      Date: 2024-01-01 to 2025-12-31
6. Categorical Uniques:    {}
7. Units Information:      Energy: MWh; Emissions/Allowances/Credits: tCO2; Carbon Price: USD/t; Costs: USD
8. Source / Provenance:    Simulated company trading & compliance transaction dataset
9. Project Role:           Company-level carbon trading
10. Granularity:           Company-transaction (daily 2024-2025)
11. Data Quality Concerns: Synthetic dataset; perfectly complete (0 nulls); future simulated dates (2024-2025); company-level granularity must not be merged directly with macroeconomic/country series.
12. Recommendation:       KEEP
13. Column Names (15): ['Company_ID', 'Industry_Type', 'Date', 'Energy_Demand_MWh', 'Fuel_Type', 'Emission_Produced_tCO2',

## 7. Audit Errors & Diagnostics
Explicitly tracks any failed reads or unhandled formats.

In [8]:
if errors_df.empty:
    print("SUCCESS: 0 audit errors. All 24 datasets and sheets were successfully parsed and profiled.")
else:
    print(f"ALERT: {len(errors_df)} audit error(s) encountered:")
    display(errors_df)

SUCCESS: 0 audit errors. All 24 datasets and sheets were successfully parsed and profiled.


## 8. Executive Summary
Final concise summary synthesized for the project pipeline.

In [9]:
total_files = len(RAW_FILES)
successful_audits = len(summary_df)
failed_audits = len(errors_df)
role_counts = summary_df["role"].value_counts().to_dict()
rec_counts = summary_df["recommendation"].value_counts().to_dict()

print("=" * 75)
print("           DATA AUDIT EXECUTIVE SUMMARY")
print("=" * 75)
print(f"Total Supported Datasets Discovered: {total_files}")
print(f"Successfully Audited (Tables/Sheets): {successful_audits}")
print(f"Failed Audits:                        {failed_audits}")
print("-" * 75)
print("Dataset Categories (Project Roles):")
for role, count in role_counts.items():
    print(f"  - {role:32s}: {count} table(s)")
print("-" * 75)
print("Recommendations Breakdown:")
for rec, count in rec_counts.items():
    print(f"  - {rec:32s}: {count} table(s)")
print("-" * 75)
print("Key Data-Quality Issues Identified:")
print("  1. Encoding Mismatch: 'world_gdp_data.csv' is Latin1 encoded (fails under UTF-8).")
print("  2. Openpyxl Compatibility: 'OECD.CTP.TPS...xlsx' has empty <fill/> XML styling elements requiring runtime patch.")
print("  3. Wide vs Long Format: 'world_gdp_data.csv' has 45 yearly columns (1980-2024); requires unpivoting (melting) to merge with country panel.")
print("  4. Incompatible Granularities: Company transactions (daily), Country panels (annual), and Global aggregates (annual) must NOT be merged blindly.")
print("  5. Extreme Historical Missingness: GCB emissions files span 1750-2021 with heavy nulls pre-1950; must filter to modern era (1990-2021).")
print("  6. Feature Redundancy: 17 renewable CSVs contain overlapping metrics across TWh, shares, and capacities.")
print("  7. Non-Numeric First Row: 'voluntary-carbon-market...xlsx' contains 'pre-2005' string in year column.")
print("-" * 75)
print("Recommended Datasets for Next Data-Cleaning Phase:")
print("  [Core ML Models]")
print("  - 'data/raw/Carbon Trading Transactions Dataset/carbon_trading_dataset.csv' (Company trading & compliance ML)")
print("  - 'data/raw/Emissions by Country/GCB2022v27_MtCO2_flat.csv' (National emissions feature engineering)")
print("  - 'data/raw/Renewable Energy World Wide  1965~2022/03 modern-renewable-prod.csv' (Renewable production TWh)")
print("  - 'data/raw/Renewable Energy World Wide  1965~2022/04 share-electricity-renewables.csv' (Renewable electricity share %)")
print("  - 'data/raw/World GDP Growth/world_gdp_data.csv' (Economic growth feature engineering after melt)")
print("  - 'data/raw/voluntary-carbon-market-size-by-value-and-volume-of-traded-carbon-credits.xlsx' (Global voluntary carbon market forecasting)")
print("  [Supporting / Context Data]")
print("  - 'data/raw/OECD.CTP.TPS...xlsx' (Static country carbon tax and ETS benchmark rates)")
print("  - Remaining renewable energy capacity/share files (selective imputation / feature enrichment)")
print("  [Exclude from ML]")
print("  - 'data/raw/Emissions by Country/GCB2022v27_sources_flat.csv' (Metadata / provenance only)")
print("=" * 75)

           DATA AUDIT EXECUTIVE SUMMARY
Total Supported Datasets Discovered: 24
Successfully Audited (Tables/Sheets): 25
Failed Audits:                        0
---------------------------------------------------------------------------
Dataset Categories (Project Roles):
  - Renewable-energy features       : 17 table(s)
  - Country/regional intelligence   : 2 table(s)
  - Carbon pricing/policy           : 2 table(s)
  - Company-level carbon trading    : 1 table(s)
  - Supporting/reference data       : 1 table(s)
  - Global market forecasting       : 1 table(s)
  - Economic features               : 1 table(s)
---------------------------------------------------------------------------
Recommendations Breakdown:
  - KEEP AS SUPPORTING DATA         : 18 table(s)
  - KEEP                            : 6 table(s)
  - DO NOT USE FOR ML               : 1 table(s)
---------------------------------------------------------------------------
Key Data-Quality Issues Identified:
  1. Encoding Mismat